# **Day 46: Object Oriented Programming (Part 6) - Multiple Inheritance**

In Python, a class can have **more than one parent**. This is called **multiple inheritance**.

**Real-life example:** A person can be both a **swimmer** and a **runner** at the same time. A triathlete gets skills from both.

In [1]:
class Swimmer:
    def swim(self):
        return f"{self.name} can swim"

class Runner:
    def run(self):
        return f"{self.name} can run"

class Triathlete(Swimmer, Runner):
    def __init__(self, name):
        self.name = name

t = Triathlete("Mihir")
print(t.swim())
print(t.run())

Mihir can swim
Mihir can run


## **The Diamond Problem**
What happens if **both parents** have a method with the **same name**? Which one does Python use?

```
         Staff
        /     \
   Teacher    Coach
        \     /
     SportsTeacher
```

In [2]:
class Staff:
    def duty(self):
        return "General staff duty"

class Teacher(Staff):
    def duty(self):
        return "Teaching"

class Coach(Staff):
    def duty(self):
        return "Coaching"

class SportsTeacher(Teacher, Coach):
    pass

print(SportsTeacher().duty())

Teaching


## **MRO: Method Resolution Order**
Python follows a fixed order to search for a method, called the **MRO (Method Resolution Order)**. The simple rule is:
1. First the class itself.
2. Then the parents **from left to right** (in the order we wrote them).
3. A parent always comes **after** all its children.

We can see the MRO with `ClassName.__mro__` or `ClassName.mro()`.

In [3]:
print([c.__name__ for c in SportsTeacher.__mro__])

['SportsTeacher', 'Teacher', 'Coach', 'Staff', 'object']


So Python checks: SportsTeacher -> **Teacher** -> Coach -> Staff -> object. It finds `duty()` in `Teacher` first, so it prints "Teaching".

## **super() Follows the MRO**
`super()` does not always mean "my parent". It means "**the next class in the MRO**". This makes sure every class runs **only once**.

In [4]:
class Staff:
    def __init__(self):
        print("Staff __init__")

class Teacher(Staff):
    def __init__(self):
        print("Teacher __init__")
        super().__init__()

class Coach(Staff):
    def __init__(self):
        print("Coach __init__")
        super().__init__()

class SportsTeacher(Teacher, Coach):
    def __init__(self):
        print("SportsTeacher __init__")
        super().__init__()

SportsTeacher()

SportsTeacher __init__
Teacher __init__
Coach __init__
Staff __init__


Notice that `Teacher`'s `super()` went to `Coach` (the next in the MRO), not to `Staff`. And `Staff` ran only **once**.

## **Mixins: The Best Use of Multiple Inheritance**
A **mixin** is a small class that adds **one extra ability** to other classes. It is not used alone.

Think of mixins like **add-on toppings** on a pizza: the base is the same, and we add what we need.

In [5]:
import json

class JsonMixin:
    """Adds the ability to turn an object into JSON text."""
    def to_json(self):
        return json.dumps(self.__dict__, ensure_ascii=False)

class PrintMixin:
    """Adds the ability to print all attributes neatly."""
    def show(self):
        for key, value in self.__dict__.items():
            print(f"  {key}: {value}")

class Sweet(JsonMixin, PrintMixin):
    def __init__(self, name, shop, price):
        self.name = name
        self.shop = shop
        self.price = price

s = Sweet("Nolen gurer sandesh", "Bhim Chandra Nag", 30)
print(s.to_json())
s.show()

{"name": "Nolen gurer sandesh", "shop": "Bhim Chandra Nag", "price": 30}
  name: Nolen gurer sandesh
  shop: Bhim Chandra Nag
  price: 30


## **Example from GIS: Map Layers with Mixins**
In GIS software like QGIS, a map is made of **layers** (roads, rivers, districts and so on). Different layers need different abilities:
- Some layers can be **exported** to GeoJSON.
- Some layers can be **styled** with a colour.

We can give these abilities using mixins.

In [6]:
class Layer:
    def __init__(self, name, features):
        self.name = name
        self.features = features          # A list of dictionaries

    def count(self):
        return len(self.features)

class ExportMixin:
    def export_geojson(self):
        return {"type": "FeatureCollection", "name": self.name,
                "features": [{"type": "Feature",
                              "geometry": {"type": "Point", "coordinates": [f["lon"], f["lat"]]},
                              "properties": {"name": f["name"]}} for f in self.features]}

class StyleMixin:
    def set_style(self, colour, size=5):
        self.style = {"colour": colour, "size": size}
        return f"Layer '{self.name}' styled with {colour}, size {size}"

class SchoolLayer(Layer, ExportMixin, StyleMixin):
    pass

schools = SchoolLayer("Schools of Midnapore", [
    {"name": "Midnapore Collegiate School", "lat": 22.4239, "lon": 87.3190},
    {"name": "Vidyasagar Vidyapith", "lat": 22.4196, "lon": 87.3172},
])

print("Features:", schools.count())
print(schools.set_style("blue", 8))
print(json.dumps(schools.export_geojson(), indent=1)[:300], "...")

Features: 2
Layer 'Schools of Midnapore' styled with blue, size 8
{
 "type": "FeatureCollection",
 "name": "Schools of Midnapore",
 "features": [
  {
   "type": "Feature",
   "geometry": {
    "type": "Point",
    "coordinates": [
     87.319,
     22.4239
    ]
   },
   "properties": {
    "name": "Midnapore Collegiate School"
   }
  },
  {
   "type": "Feature",
 ...


Let us understand the code above:
- `SchoolLayer` gets `count()` from `Layer`, `export_geojson()` from `ExportMixin`, and `set_style()` from `StyleMixin`.
- Each mixin does **one small job**. We can mix them into any other layer class too (roads, hospitals, and so on).

## **When the MRO Cannot Be Made**
Some class orders are impossible, and Python refuses them.

In [7]:
class A: pass
class B(A): pass

# class C(A, B): pass     # This will raise a TypeError (A cannot come before its own child B)

## **Tip**
Use multiple inheritance mostly for **mixins**. If it becomes confusing, it is usually better to use **composition** (putting objects inside objects) instead.

In [8]:
# Quick question
# What will be printed?
# class X:
#     def f(self): return "X"
# class Y:
#     def f(self): return "Y"
# class Z(Y, X): pass
# print(Z().f())